supporter code

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
url = "https://topforeignstocks.com/indices/components-of-the-sp-500-index/"
r = requests.get(url)
soup = BeautifulSoup(r.content, "html.parser")
listt = []
for i in soup.find_all('a', href=True):
  if str(i).find("noopener") != -1:
      listt.append(str(i))
ticker_list = []
for i in listt:
  ticker_list.append(i.split(">")[1][0:len(i.split(">")[1])-3])
#file 1
stock_list1 = pd.read_csv("/content/sample_data/betterer list")
#file 2
stock_list2 = pd.read_csv("/content/sample_data/all_stocks_5yr.csv")
stock_list2 = stock_list2.rename(columns={"Name": "Ticker"})
stock_list2["diff"] = stock_list2["close"] - stock_list2["open"]
for i in stock_list2["Ticker"].unique():
  if i not in ticker_list:
    ticker_list.append(i)

In [ ]:
data1= get_data('AMZN')
data2= get_data2("AAPL")
n = data1['Close'].corr(data2['Close'])
n


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


0.27747237541908476

In [ ]:
def get_data2(ticker):
  data = yf.download(ticker, start="2019-01-01", end="2019-12-31")
  return data

In [ ]:
import itertools as it
import yfinance as yf
import pandas as pd
def get_data(ticker):
  data = yf.download(ticker, start="2019-01-01", end="2022-12-31")
  return data
def similar_to_2019to2023(ticker):
# List of stocks
  stocks = ticker_list

# Get all possible combinations of stocks
#combinations = list(it.combinations(stocks, 2))

# Initialize an empty data frame to store the results
  result = pd.DataFrame(columns=['stock1', 'stock2', 'correlation'])
  data2 = yf.download(ticker, start="2019-01-01", end="2022-12-31")
# Loop over all combinations of stocks and calculate their correlation
  for i in range(len(stocks)):
    stock1 = stocks[i]
    try:
      data1 = yf.download(stock1, start="2019-01-01", end="2022-12-31")
      correlation = data1['Close'].corr(data2['Close'])
    except KeyError:
      correlation = 0
    if stock1 != ticker:
      result.loc[i] = [stock1, ticker, correlation]

# Sort the results by correlation in descending order
  result = result.sort_values('correlation', ascending=False)
  return result

In [ ]:
import numpy as np
def similar_to_2013to2018(ticker):
  reference = stock_list2[stock_list2["Ticker"] == ticker]
  tickers = []
  corrs = []
  for i in stock_list2["Ticker"].unique():
    if i != ticker:
      compare = stock_list2[stock_list2["Ticker"] == i]
      a = (np.array(compare['close']))
      b = (np.array(reference['close']))
      try:
        corr = np.corrcoef(a, b)[0, 1]
        corrs.append(corr)
      except ValueError:
        corrs.append(np.NaN)
      tickers.append(i)
        
  dataa = {'Ticker':tickers,
        'Correlation':corrs}
  ranked_df = pd.DataFrame(dataa)
  ranked_df = ranked_df.sort_values('Correlation', ascending=False)
  return ranked_df

In [ ]:
#comp name from ticker
import requests
def name_search(ticker):  
  url = 'https://www.alphavantage.co/query?function=SYMBOL_SEARCH&keywords=' + str(ticker) + '&apikey=8O4UQOTLNXP3DQJV'
  r = requests.get(url)
  data = r.json()
  return data
def company_name(ticker):
  result = name_search(ticker)
  try:
    if result["bestMatches"] == []:
      return ""
    else:
      best = str(result["bestMatches"][0]).split(",")
      g = best[1].split(":")[1]
      company_name = g[2:len(g)-1]
      return company_name
  except KeyError:
    return ""

    

In [ ]:
import yfinance as yf
import requests

def info0(ticker):
  info = yf.Ticker(ticker)
  return info.info

def info1(company_name):
  url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
  r = requests.get(url)
  data = r.text
  marker = data.find(company_name)
  if marker == -1:
    marker = data.find(str(company_name[0:len(company_name)-4]))
    if marker == -1:
      info_list = []
      return info_list
  raw_data = data[marker-10:marker+355]
  thing = raw_data.split("<td>")
  try:
    sector = thing[1][0:len(thing[1])-7]
  except IndexError:
    sector = ""
  try:
    subindustry = thing[2][0:len(thing[1])-7]
  except IndexError:
    subindustry = ""
  try:
    location = thing[3].split(",")[2].split(">")[1] + "," + thing[3].split(",")[2].split(">")[0]
    location = location[0:len(location)-1]
  except IndexError:
    location = ""
  try:
    Date_Added = thing[4].split("<")[0]
  except IndexError:
    Date_Added = ""
  try:
    CKS = thing[5].split("<")[0]
  except IndexError:
    CKS = ""
  try:
    founded = thing[6][0:4]
  except IndexError:
    founded = ""
  info_list = [sector, subindustry, location, Date_Added, CKS, founded]
  return info_list

def info2(string, ticker):
  stri = ">" + str(ticker) + "<"
  num = string.find(stri)
  needed = ""
  stop = 0
  while stop < 1:
    if string[num] + string[num+1] + string[num+2] + string[num+3] + string[num+4] == "href=":
      stop += 1
    else:
      needed += string[num] 
      num += 1
  return(needed)

#call with requests.get("https://stockanalysis.com/stocks/").text, ticker)
def search_engine(ticker):
  comp_name = company_name(ticker)
  list_check1 = info0(ticker)
  if list_check1 == []:
    list_check2 = info1(comp_name)
    if list_check2 == []:
      list_check3 = info2(requests.get("https://stockanalysis.com/stocks/").text, ticker)
      if list_check3 == []:
        info = ""
      else:
        info = list_check3
    info = list_check2
  info = list_check1
    #use_list should have sector, subindustry, location, date_added, CKS, Founded
  company_info = {
      "Ticker": ticker,
      "Company": comp_name,
      "Info": info
  }
  return company_info

In [ ]:
#maybe 
def info2(string, ticker):
  stri = ">" + str(ticker) + "<"
  num = string.find(stri)
  needed = ""
  stop = 0
  while stop < 1:
    if string[num] + string[num+1] + string[num+2] + string[num+3] + string[num+4] == "href=":
      stop += 1
    else:
      needed += string[num] 
      num += 1
  return(needed)

#call with requests.get("https://stockanalysis.com/stocks/").text, ticker)

def search_engine(ticker):
  comp_name = company_name(ticker)
  list_check1 = info0(ticker)
  if list_check1 == []:
    list_check2 = info1(comp_name)
    if list_check2 == []:
      list_check3 = info2(requests.get("https://stockanalysis.com/stocks/").text, ticker)
      if list_check3 == []:
        info = ""
      else:
        info = list_check3
    info = list_check2
  info = list_check1
    #use_list should have sector, subindustry, location, date_added, CKS, Founded
  company_info = {
      "Ticker": ticker,
      "Company": comp_name,
      "Info": info
  }
  return company_info

end of supporting code

database building 

1st iteration 

In [ ]:
#listt = [ticker_list[0], company_name(ticker_list[0]), info1(ticker_list[0]), data]
database = pd.DataFrame(columns = ["Ticker", "Company Name", "Info", "Data"])

In [ ]:
info_list = []
for i in ticker_list:
  info_list.append(info1(i))

KeyboardInterrupt: ignored

In [ ]:
companies = []
for i in ticker_list:
  companies.append(company_name(i))

In [ ]:
data_list = []
for i in ticker_list:
  data = get_data(i)
  dates = data.index
  open = data["Open"]
  close = data["Close"]
  adj_close = data["Adj Close"]
  volume = data["Volume"]
  data_dict = {"dates": dates, "open": open, "close": close, "adj_close": adj_close, "volume": volume}
  data_list.append(data_dict)

In [ ]:
database["Ticker"] = ticker_list
database["Company Name"] = companies
database["Info"] = info_list
database["Data"] = data_list
database = database.set_index("Ticker")

2nd iteration 

In [ ]:
import pandas as pd
database = pd.read_csv("/content/sample_data/companyinfo.csv")
a = 0
listt = []
while a < len(database):
  entry = database.iloc[a]
  if str(entry["Info"]) == "['', '', '', '', '', '']":
    listt.append(a)
  a += 1
database = database.drop(listt)
import numpy as np
a = 0
new_comp_name_list = []
while a < len(database):
  entry = database.iloc[a]
  if str(entry["Company Name"]) == 'nan':
    if str(entry["Info"][0:3]) != "[''":
      try:
        d = entry["Info"].split("=")[1].split("/")[2].split(" ")[0]
        add = d[0:len(d)-1]
        new_comp_name_list.append(add)
      except IndexError:
        new_comp_name_list.append("")
    else:
      new_comp_name_list.append("")
  else:
    new_comp_name_list.append(entry["Company Name"])
  a += 1
database["Company Name"] = new_comp_name_list

data reference, eliminate current missing info values that arent real

In [ ]:
a = 0
listt = []
while a < len(database):
  entry = database.iloc[a]
  if str(entry["Data"])[0:18] == "{'dates': Index([]":
    if a != 495 and a != 510 and a != 514 and a != 516 and a != 570:
      listt.append(a)
  elif len(str(entry["Info"])) < 48:
    listt.append(a)
  a += 1
database = database.drop(listt)

In [ ]:
database["Company Name"].value_counts(normalize=True)

In [ ]:
empty = database[database["Company Name"] == ""]
drop_list = []
b = 0
while b < len(empty):
  index = str(empty.iloc[b]["Data"]).find("DatetimeIndex")
  if index == -1:
    drop_list.append(empty.iloc[b]["Ticker"])
  b += 1

In [ ]:
database = database.set_index("Ticker")
database = database.drop(drop_list)
database

In [ ]:
database.to_csv("Fcompany info.csv")

In [ ]:
database = pd.read_csv("/content/sample_data/Fcompany info.csv")

In [ ]:
database_2 = database.reset_index(drop=False)
database_2.iloc[0]["Info"]

'[\'<a href="/wiki/Apple_Inc." title="Apple Inc.">Apple Inc.</a\', \'Information Technology</td>\\n\', \'\', \'\', \'1982-11-30\', \'0000\']'

end of database section

search engine

In [ ]:
#specify ticker
ticker = "AMZN"

def search_engine(ticker):
  comp_name = company_name(ticker)
  list_check1 = info0(ticker)
  if list_check1 == []:
    list_check2 = info1(comp_name)
    if list_check2 == []:
      list_check3 = info2(requests.get("https://stockanalysis.com/stocks/").text, ticker)
      if list_check3 == []:
        info = ""
      else:
        info = list_check3
    info = list_check2
  info = list_check1
    #use_list should have sector, subindustry, location, date_added, CKS, Founded
  company_info = {
      "Ticker": ticker,
      "Company": comp_name,
      "Info": info
  }
  return company_info

In [ ]:
#2019 to 2023 comparison:
company_comparison_2019to2023 = similar_to_2019to2023(ticker)
search_engine(ticker)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

1 Failed download:
- BRK.B: No timezone found, symbol may be delisted
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%******

{'Ticker': 'AMZN',
 'Company': 'Amazon.com Inc',
 'Info': {'address1': '410 Terry Avenue North',
  'city': 'Seattle',
  'state': 'WA',
  'zip': '98109-5210',
  'country': 'United States',
  'phone': '206 266 1000',
  'website': 'https://www.amazon.com',
  'industry': 'Internet Retail',
  'industryDisp': 'Internet Retail',
  'sector': 'Consumer Cyclical',
  'longBusinessSummary': "Amazon.com, Inc. engages in the retail sale of consumer products and subscriptions through online and physical stores in North America and internationally. It operates through three segments: North America, International, and Amazon Web Services (AWS). The company's products offered through its stores include merchandise and content purchased for resale; and products offered by third-party sellers. It also manufactures and sells electronic devices, including Kindle, Fire tablets, Fire TVs, Rings, Blink, eero, and Echo; and develops and produces media content. In addition, the company offers programs that enabl

In [ ]:
search_engine("ANSS")

{'Ticker': 'ANSS',
 'Company': 'Ansys Inc',
 'Info': {'address1': '2600 ANSYS Drive',
  'city': 'Canonsburg',
  'state': 'PA',
  'zip': '15317',
  'country': 'United States',
  'phone': '844 462 6797',
  'fax': '724 514 9494',
  'website': 'https://www.ansys.com',
  'industry': 'Software—Application',
  'industryDisp': 'Software—Application',
  'sector': 'Technology',
  'longBusinessSummary': 'ANSYS, Inc. develops and markets engineering simulation software and services worldwide. It offers ANSYS Workbench, a framework upon which its multiphysics engineering simulation technologies are built and enables engineers to simulate the interactions between structures, heat transfer, fluids, electronics, and optical elements in a unified engineering simulation environment; high-performance computing product suite and the cloud; power analysis and optimization software suite that manages the power budget, power delivery integrity, and power-induced noise in an electronic design; and structural 

In [ ]:
company_comparison_2019to2023

,stock1,stock2,correlation
258,ANSS,AMZN,0.926332
36,ADBE,AMZN,0.925531
451,BIO,AMZN,0.913093
77,NOW,AMZN,0.910133
349,SWKS,AMZN,0.909133
...,...,...,...
593,TIF,AMZN,NaN
594,TMK,AMZN,NaN
596,TSS,AMZN,NaN
597,TWX,AMZN,NaN


In [ ]:
#2013 to 2018 comparison:
company_comparison_2013to2018 = similar_to_2013to2018(ticker)
company_comparison_2013to2018

,Ticker,Correlation
49,ATVI,0.978756
41,AOS,0.976013
52,AVY,0.974645
205,GOOGL,0.973988
6,ACN,0.971033
...,...,...
388,QRVO,NaN
429,SYF,NaN
454,UA,NaN
464,UTX,NaN


In [ ]:
get_data(ticker)

[*********************100%***********************]  1 of 1 completed


,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,
2019-01-02,73.260002,77.667999,73.046501,76.956497,76.956497,159662000
2019-01-03,76.000504,76.900002,74.855499,75.014000,75.014000,139512000
2019-01-04,76.500000,79.699997,75.915497,78.769501,78.769501,183652000
2019-01-07,80.115501,81.727997,79.459503,81.475502,81.475502,159864000
2019-01-08,83.234497,83.830498,80.830498,82.829002,82.829002,177628000
...,...,...,...,...,...,...
2022-12-23,83.250000,85.779999,82.930000,85.250000,85.250000,57433700
2022-12-27,84.970001,85.349998,83.000000,83.040001,83.040001,57284000
2022-12-28,82.800003,83.480003,81.690002,81.820000,81.820000,58228600


In [ ]:
search_engine(ticker)

{'Ticker': 'AMZN',
 'Company': 'Amazon.com Inc',
 'Info': {'address1': '410 Terry Avenue North',
  'city': 'Seattle',
  'state': 'WA',
  'zip': '98109-5210',
  'country': 'United States',
  'phone': '206 266 1000',
  'website': 'https://www.amazon.com',
  'industry': 'Internet Retail',
  'industryDisp': 'Internet Retail',
  'sector': 'Consumer Cyclical',
  'longBusinessSummary': "Amazon.com, Inc. engages in the retail sale of consumer products and subscriptions through online and physical stores in North America and internationally. It operates through three segments: North America, International, and Amazon Web Services (AWS). The company's products offered through its stores include merchandise and content purchased for resale; and products offered by third-party sellers. It also manufactures and sells electronic devices, including Kindle, Fire tablets, Fire TVs, Rings, Blink, eero, and Echo; and develops and produces media content. In addition, the company offers programs that enabl

In [ ]:
key = input("Choose what information you want from this list:" + str(search_engine(ticker)["Info"].keys()))
print(search_engine(ticker)["Info"][str(key)])

In [ ]:
import pandas as pd
import yfinance as yf

In [ ]:
def createList(r1, r2):
 if (r1 == r2):
  return r1
 else:
  res = []
  while(r1 < r2):   
   res.append(r1)
   r1 += 1
  return res 
def similar(ticker, start_year, end_year):
  stocks = ["AMZN", "AFL"]
  listt = createList(start_year, end_year)
# reference stock, inputted ticker, is set to recent 4 years (2019 to 2023)
  result = pd.DataFrame(columns=['stock1', 'stock2', 'correlation'])
  data2 = yf.download(ticker, start="2019-01-01", end="2022-12-31")
  for i in range(len(stocks)):
    stock1 = stocks[i]
# Loop over all combinations of stocks and calculate their correlation
    #str(start_year) str(end_year)
    for year in listt:
      try:
        data1 = yf.download(stock1, start=str(year)+"-01-01", end=str(year)+"-12-31")
        correlation = data1['Close'].corr(data2['Close'])
      except KeyError:
        correlation = 0
      if stock1 != ticker:
        result.loc[len(result)] = [str(year)+stock1, ticker, correlation]
  result = result.sort_values('correlation', ascending=False)
  return result


In [ ]:
AAPL = similar('AAPL', 2018, 2023)
AAPL

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


,stock1,stock2,correlation
2,2020AMZN,AAPL,0.921036
4,2022AMZN,AAPL,0.896738
3,2021AMZN,AAPL,0.617575
6,2019AFL,AAPL,0.597924
8,2021AFL,AAPL,0.499789
1,2019AMZN,AAPL,0.277472
9,2022AFL,AAPL,0.088952
7,2020AFL,AAPL,0.005956
0,2018AMZN,AAPL,NaN
5,2018AFL,AAPL,NaN


In [ ]:
import pandas as pd
stock_list2 = pd.read_csv("/content/sample_data/all_stocks_5yr.csv")
import numpy as np

In [ ]:
a = 0
yeared_tickers = []
while a < len(stock_list2):
  entry = stock_list2.iloc[a]
  year = entry["date"][0:4]
  yeared_tickers.append(str(year) + str(entry["Name"]))
  a += 1
stock_list2["Ticker"] = yeared_tickers

In [ ]:
stock_list2

,date,open,high,low,close,volume,Name,Ticker
0,2013-02-08,15.07,15.12,14.630,14.75,8407500,AAL,2013AAL
1,2013-02-11,14.89,15.01,14.260,14.46,8882000,AAL,2013AAL
2,2013-02-12,14.45,14.51,14.100,14.27,8126000,AAL,2013AAL
3,2013-02-13,14.30,14.94,14.250,14.66,10259500,AAL,2013AAL
4,2013-02-14,14.94,14.96,13.160,13.99,31879900,AAL,2013AAL
...,...,...,...,...,...,...,...,...
570227,2013-12-20,109.22,111.13,109.070,110.66,4618000,UTX,2013UTX
570228,2013-12-23,111.42,111.47,110.570,110.82,1972776,UTX,2013UTX
570229,2013-12-24,110.80,111.60,110.684,111.47,747902,UTX,2013UTX
570230,2013-12-26,111.61,112.76,111.560,112.69,2449588,UTX,2013UTX


In [ ]:
def similar2013(ticker):
  reference = stock_list2[stock_list2["Name"] == ticker]
  tickers = []
  corrs = []
  #for i in stock_list2["Ticker"].unique():
  for i in ["AMZN", "AAL"]:
    if i != ticker:
      compare = stock_list2[stock_list2["Name"] == i]
      for g in compare["Ticker"].unique():
        use = compare[compare["Ticker"] == g]
        #print((use['close']))
        #print((reference['close']))
        #a = (use['close']).corr((reference['close']))
        a = np.cov(use['close'],reference['close'])/[np.stdev(use['close'])*np.stdev(reference['close'])]
        print(a)
        
        

In [ ]:
similar2013("AAPL")

ValueError: ignored